<a href="https://colab.research.google.com/github/ValNR/IntroIA/blob/main/04%20-%20modelo_gradient_boosting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:

# 04 - MODELO GRADIENT BOOSTING (XGBoost)

import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.model_selection import cross_val_score
import warnings
warnings.filterwarnings('ignore')

# Cargar datos
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

# Separar componentes
train_ids = train['ID'].copy()
test_ids = test['ID'].copy()
y_train = train['RENDIMIENTO_GLOBAL'].copy()

X_train = train.drop(['ID', 'RENDIMIENTO_GLOBAL'], axis=1)
X_test = test.drop(['ID'], axis=1)

# Identificar columnas
numeric_cols = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_cols = X_train.select_dtypes(include=['object']).columns.tolist()

# Limpieza
for col in categorical_cols:
    X_train[col] = X_train[col].replace(['', ' ', 'nan'], np.nan)
    X_test[col] = X_test[col].replace(['', ' ', 'nan'], np.nan)

# Imputación
if len(numeric_cols) > 0:
    num_imp = SimpleImputer(strategy='median')
    X_train[numeric_cols] = num_imp.fit_transform(X_train[numeric_cols])
    X_test[numeric_cols] = num_imp.transform(X_test[numeric_cols])

if len(categorical_cols) > 0:
    cat_imp = SimpleImputer(strategy='most_frequent')
    X_train[categorical_cols] = cat_imp.fit_transform(X_train[categorical_cols])
    X_test[categorical_cols] = cat_imp.transform(X_test[categorical_cols])

# Codificación
for col in categorical_cols:
    le = LabelEncoder()
    combined = pd.concat([X_train[col].astype(str), X_test[col].astype(str)])
    le.fit(combined)
    X_train[col] = le.transform(X_train[col].astype(str))
    X_test[col] = le.transform(X_test[col].astype(str))

# Limpieza final
X_train = X_train.replace([np.inf, -np.inf], np.nan).fillna(0)
X_test = X_test.replace([np.inf, -np.inf], np.nan).fillna(0)

# Codificar target
le_target = LabelEncoder()
y_train_encoded = le_target.fit_transform(y_train)

# Entrenar XGBoost
xgb_model = XGBClassifier(
    n_estimators=200,
    max_depth=8,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    objective='multi:softmax',
    num_class=4,
    random_state=42,
    n_jobs=-1
)

print('Entrenando XGBoost...')
xgb_model.fit(X_train, y_train_encoded)
print(' XGBoost entrenado')

# Evaluación
train_score = xgb_model.score(X_train, y_train_encoded)
print(f'Accuracy en train: {train_score:.4f}')

# Validación cruzada
cv_scores = cross_val_score(xgb_model, X_train, y_train_encoded, cv=3)
print(f'CV Accuracy: {cv_scores.mean():.4f}')

# Predicciones
predictions_encoded = xgb_model.predict(X_test)
predictions = le_target.inverse_transform(predictions_encoded)

# Feature Importance
feature_importance = pd.DataFrame({
    'feature': X_train.columns,
    'importance': xgb_model.feature_importances_
}).sort_values('importance', ascending=False)

print('\nTop 10 Features:')
print(feature_importance.head(10))

# Submission
submission = pd.DataFrame({
    'ID': test_ids,
    'RENDIMIENTO_GLOBAL': predictions
})

submission.to_csv('submission_xgboost.csv', index=False)
print('\n Archivo submission_xgboost.csv generado ')

Entrenando XGBoost...
 XGBoost entrenado
Accuracy en train: 0.4902
CV Accuracy: 0.4327

Top 10 Features:
                        feature  importance
3   E_VALORMATRICULAUNIVERSIDAD    0.177893
5             F_ESTRATOVIVIENDA    0.118196
11        E_PAGOMATRICULAPROPIO    0.099557
13            F_TIENEINTERNET.1    0.070717
12            F_TIENECOMPUTADOR    0.061643
1              E_PRGM_ACADEMICO    0.059967
6               F_TIENEINTERNET    0.052852
14             F_EDUCACIONMADRE    0.050846
2           E_PRGM_DEPARTAMENTO    0.049198
15                  INDICADOR_1    0.044979

 Archivo submission_xgboost.csv generado 
